# 48. 点图（pointplot）

<!-- module-learning-arc:start -->
> **Seaborn 模块主线｜第 5 / 20 步：比较类别频数、水平与组内分布**
>
> **持续应用背景：** 开展客群消费行为差异研究：先固定样本和统计语义，再比较分布、关系和分面结果，判断差异是否稳定。
>
> **承接上一阶段：** 统计柱状图（barplot）  →  **本章任务：** 点图（pointplot）  →  **下一步：** 箱线图（boxplot）
>
> **大作业连接：** 本章练习将成为《客群消费行为差异研究》的一部分，最终需要从样本口径和分布比较走到关系验证、分面研究与因果边界说明。
<!-- module-learning-arc:end -->


## 本章场景

现实中很多分析问题都归结为一句话——不同分组在某个指标上谁表现得更好。



## 本章目标

学完本章，你将能够：

- **理解**：理解「点图（pointplot）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「点图（pointplot）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「点图（pointplot）」并读出其中的结论。


## 48.1 适用场景

**背景引入**：现实中很多分析问题都归结为一句话——不同分组在某个指标上谁表现得更好。柱状图用高度表达这一点，可一旦分组很多，或想在两组之间连一条线观察变化趋势，柱图就显得拥挤。点图（pointplot）把每个分组浓缩成一个均值点和一条误差带，用很小的图形面积就讲清楚了“哪个高、怎么变”，是快速比较两个分类变量的首选。

打个比方：pointplot 像'给一组选手的排名打点'——每人一个圆点和一段上下浮动的小杠，你关心的不是'这根柱子有多高'，而是'这些点从左到右怎么串、串起来是上坡还是下坡'。分组一多、想在组间连出趋势线时，它比柱图轻巧得多。

比较多个类别或两个因素下的均值趋势，不需要柱形面积。


## 48.2 数据结构

一至两个分类变量和一个数值变量。


## 48.3 本章练习任务

运行基础图表后，完成以下任务：

1. 将 dodge=0.25 改为 dodge=0，观察分组错位与重叠显示的差异
2. 修改 markers 参数从 ["o", "s", "^"] 为 ["D", "v", "p"]，对比不同标记形状的区分度
3. 调整 linestyles 从 ["-", "--", ":"] 为全部 "-"，说明线型对多组区分的作用


## 48.4 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `plt.subplots()`、`sns.pointplot()`、`ax.set()`、`fig.tight_layout()` | 比较多个类别或两个因素下的均值趋势，不需要柱形面积。 | 把类别间连接线解释为连续时间 |
| 进阶变体 | `plt.subplots()`、`sns.pointplot()`、`ax.set()`、`ax.legend()` | 在基础图表上增加分组、注释、布局或交互 | 类别顺序没有业务意义 |
| 关键参数 | `estimator` | 点估计 | 把类别间连接线解释为连续时间 |
| 关键参数 | `errorbar` | 误差 | 类别顺序没有业务意义 |
| 关键参数 | `dodge` | 分组错位 | 多组线条难以辨认 |
| 关键参数 | `markers/linestyles` | 样式 | 把类别间连接线解释为连续时间 |


## 48.5 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-48 -->
### 数学推导｜均值的不确定性区间

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜样本均值存在抽样波动。** 独立同分布条件下 $\operatorname{Var}(\bar X)=\sigma^2/n$。

**第 2 步｜用样本标准差估计未知的 $\sigma$。** 得到 $SE\approx s/\sqrt n$。

**第 3 步｜用标准化分布给出区间。** 大样本近似下

$$
\frac{\bar X-\mu}{SE}\approx N(0,1)
$$

标准正态中约 95% 落在 $[-1.96,1.96]$，移项后得到 $\bar x\pm1.96SE$。小样本时应把 1.96 换成相应的 $t$ 分位数。

**把上面的关系收束为本章计算式：**

$$
CI_{95\%}\approx \bar{x}\pm1.96\frac{s}{\sqrt{n}}
$$

**符号解释：** $\bar{x}$ 是样本均值，$s$ 是样本标准差，$n$ 是样本量。

**代码对应：** 统计图中的误差线应明确表示标准差、标准误还是置信区间。

**使用边界：** 该近似依赖样本与分布条件；小样本或偏态数据可考虑 bootstrap。


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，即使 seaborn 的 sns.set_theme
#      会重置字体，运行时也会在 set_theme 之后自动恢复。因此这里无需手动
#      import 或 addfont，直接使用即可。

# 1️⃣ 主题与数据导入：统一画风，读取三个公开数据集
sns.set_theme(style="whitegrid", context="notebook")

diamonds = pd.read_csv("/datasets/diamonds.csv")
taxis = pd.read_csv("/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
flights = pd.read_csv("/datasets/flights.csv")
print(
    f"Diamonds {
        len(diamonds):,    } | Taxis {
            len(taxis):,        } | Flights {
                len(flights):,            } 行"
)


In [ ]:
# 2️⃣ 特征工程：把原始字段映射成图表统一使用的列名与派生指标
orders_full = diamonds.assign(
    category=diamonds["cut"],
    channel=diamonds["color"],
    region=diamonds["clarity"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    satisfied=np.where(
        diamonds["price"] >= diamonds["price"].median(),
        "高于中位价",
        "不高于中位价",
    ),
)
orders = orders_full.sample(2_000, random_state=36)

marketing_full = taxis.assign(
    channel=taxis["payment"].fillna("unknown"),
    visits=taxis["distance"],
    ad_spend=taxis["tip"],
    sales=taxis["total"],
    conversion=(taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(
    min(2_000, len(marketing_full)), random_state=36
).copy()

daily = flights.assign(
    date=pd.to_datetime(
        flights["year"].astype(str) + "-" + flights["month"] + "-01"
    ),
    region="AirPassengers",
    sales=flights["passengers"],
)
print(
    f"样本：orders {
        len(orders):,    } | marketing {
            len(marketing):,        } | daily {
                len(daily):,            } 行"
)


## 48.6 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.2))
sns.pointplot(
    data=orders,
    x="category",
    y="order_value",
    errorbar=("ci", 90),
    color="#1a73e8",
    ax=ax,
)
ax.set(title="品类客单价点估计", xlabel="品类", ylabel="平均客单价（元）")
fig.tight_layout()
plt.show()


**练一练**：把基础图表里点图的纵轴字段从 `order_value`（品类的平均客单价）换成 `items`（平均件数），运行后看看哪个品类排名最高、排名次序会不会变。除此之外，你还可以把 `errorbar=("ci", 90)` 改成 `("ci", 68)` 或 `None`，观察误差带的宽窄。先在下方代码里补全对 `y` 字段的修改并自检，再运行观察变化。


In [ ]:
# 请在下方填写代码
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(8, 4.2))
sns.pointplot(
    data=orders,
    x="category",
    y="order_value",  # TODO: 把这里改成 "items"，看看品类排名会不会变化
    errorbar=("ci", 90),
    color="#1a73e8",
    ax=ax,
)
ax.set(title="品类客单价点估计", xlabel="品类", ylabel="平均客单价（元）")
fig.tight_layout()
plt.show()


In [ ]:
# ===== 参考实现：把纵轴从 order_value 换成 items =====


## 48.7 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.pointplot(
    data=orders,
    x="category",
    y="order_value",
    hue="channel",
    dodge=0.25,
    markers=["o", "s", "^"],
    linestyles=["-", "--", ":"],
    ci=None,
    palette="colorblind",
    ax=ax,
)
ax.set(title="渠道与品类客单价模式", xlabel="品类", ylabel="平均客单价（元）")
ax.legend(title="渠道", frameon=False)
fig.tight_layout()
plt.show()


## 48.8 参数说明

- estimator：点估计
- errorbar：误差
- dodge：分组错位
- markers/linestyles：样式


## 48.9 结果解读

读取点的位置和组间斜率；非平行线可能提示因素交互。


## 48.10 本章实训：分组比较与不确定性

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="region", y="sales", ci=None, ax=ax, color="#0F766E"
)
ax.set_title("地区销售额比较")
ax.set_ylabel("销售额")
plt.show()


### 48.10.1 第一个结果怎么读

Seaborn 负责把 DataFrame 的字段映射为图形编码；先明确横轴、纵轴和每行数据的粒度，再选择图表。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
report = report.sort_values("sales", ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="sales", y="region", ci=None, ax=ax, color="#F59E0B"
)
ax.set_title("按销售额排序的地区比较")
ax.set_xlabel("销售额")
ax.set_ylabel("地区")
plt.show()


### 48.10.2 第二个结果怎么读

第二个实验只改变排序和坐标方向，让读者更容易找到最大值。图表调整必须服务于阅读任务。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 48.11 错误恢复：分组字段缺失怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
required = {"region", "sales"}
missing = required - set(report.columns)
if missing:
    print("缺少字段：", sorted(missing))
else:
    fig, ax = plt.subplots(figsize=(6, 3))
    sns.barplot(data=report, x="region", y="sales", ci=None, ax=ax)
    ax.set_title("地区销售额")
    plt.show()


### 48.11.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

绘图前先检查字段是否存在。把字段检查放在画图之前，错误会更接近真正原因，也更容易恢复。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 48.12 易错点提醒

- 把类别间连接线解释为连续时间
- 类别顺序没有业务意义
- 多组线条难以辨认


## 48.13 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 48.14 独立迁移练习

修改一个分组、排序或统计设置，并比较修改前后的结论。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# 独立迁移练习：加入 hue 分组，比较不同渠道的点估计
# 【目标】用颜色再分一层，看不同渠道的估计是否存在差异。
import matplotlib.pyplot as plt
import seaborn as sns

# 起点示例(已可运行)：加 hue="channel"，用不同颜色区分渠道的点估计。
fig, ax = plt.subplots(figsize=(8, 4.2))
sns.pointplot(
    data=orders,
    x="category",
    y="order_value",
    hue="channel",
    errorbar=("ci", 90),
    ax=ax,
)
ax.set(title="分渠道客单价点估计", xlabel="品类", ylabel="平均客单价（元）")
ax.legend(title="渠道", frameon=False)
fig.tight_layout()
plt.show()

# ---- 反思记录：分组后，点与线的对比说明了什么 ----
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print(f"改动：{change_note}")
print(f"预期：{expected_change}")
print(f"观察：{observed_change}")


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.2))
sns.pointplot(
    data=orders,
    x="region",
    y="items",
    hue="satisfied",
    dodge=0.2,
    ci=None,
    palette=["#188038", "#f9ab00"],
    ax=ax,
)
ax.set(title="区域购买件数与评价", xlabel="区域", ylabel="平均件数")
ax.legend(title="评价", frameon=False)
fig.tight_layout()
plt.show()


## 48.15 小结

用点和连接线比较类别估计值，突出差异方向与交互模式。


### 48.15.1 你已经掌握

- 判断点图（pointplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 48.15.2 关键参数

| 参数 | 作用 |
| --- | --- |
| `estimator` | 点估计 |
| `errorbar` | 误差 |
| `dodge` | 分组错位 |
| `markers/linestyles` | 样式 |


### 48.15.3 需要注意

- 把类别间连接线解释为连续时间
- 类别顺序没有业务意义
- 多组线条难以辨认


### 48.15.4 完成检查

- [ ] 能判断什么问题适合使用点图（pointplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 48.15.5 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
